# 📋 Horizon Books — Planning in Fabric IQ Exploration

This notebook demonstrates **Planning in Fabric IQ** concepts applied to Horizon Books:
budget targets, scenario modeling, plan-vs-actual variance analysis, and writeback-ready
planning tables.

It runs **locally** against the sample CSV data and the same forecast outputs used in
`notebooks/04_Forecasting.py`.

## Planning Models

| # | Model | Table | Scenarios | Description |
|---|-------|-------|-----------|-------------|
| 1 | Revenue Targets | `PlanRevenueTargets` | Base / Optimistic / Conservative | Channel revenue targets grounded on forecasts |
| 2 | Financial Plan | `PlanFinancialTargets` | Base / Optimistic / Conservative | P&L account targets bridging budget + forecast |
| 3 | Workforce Plan | `PlanWorkforceTargets` | Base / Optimistic / Conservative | Headcount & payroll targets by department |
| 4 | Plan vs Actual | `PlanVarianceAnalysis` | — | Consolidated variance analysis with status flags |
| 5 | Scenario Summary | `PlanScenarioSummary` | All | Executive scenario comparison dashboard table |

Reference: [Planning in Fabric IQ announcement](https://blog.fabric.microsoft.com/en-us/blog/introducing-planning-in-microsoft-fabric-iq-from-historical-data-to-forecasting-the-future)

In [ ]:
# Cell 1: Setup — imports, configuration
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
from pathlib import Path
from dateutil.relativedelta import relativedelta

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

# ── Planning Configuration ──
PLANNING_HORIZON = 12         # months ahead
BASE_FISCAL_YEAR = "FY2026"
SCENARIOS = ["Base", "Optimistic", "Conservative"]
GROWTH = {"Base": 0.08, "Optimistic": 0.15, "Conservative": 0.03}
VARIANCE_THRESHOLDS = {"favorable": 0.05, "unfavorable": -0.05, "critical": -0.15}

# ── Paths ──
ROOT = Path("..")  # repo root relative to Planning/
DATA = ROOT / "SampleData"

# ── Color palette ──
SCENARIO_COLORS = {"Base": "#2E86AB", "Optimistic": "#44BBA4", "Conservative": "#E94F37"}
DOMAIN_COLORS = {"Revenue": "#2E86AB", "Finance": "#A23B72", "Workforce": "#F18F01"}

print(f"Planning horizon   : {PLANNING_HORIZON} months")
print(f"Base fiscal year   : {BASE_FISCAL_YEAR}")
print(f"Scenarios          : {', '.join(SCENARIOS)}")
print(f"Growth assumptions : {GROWTH}")
print(f"Data path          : {DATA.resolve()}")

In [ ]:
# Cell 2: Load sample data
df_orders    = pd.read_csv(DATA / "Operations" / "FactOrders.csv", parse_dates=["OrderDate"])
df_books     = pd.read_csv(DATA / "Operations" / "DimBooks.csv")
df_budget    = pd.read_csv(DATA / "Finance" / "FactBudget.csv")
df_finance   = pd.read_csv(DATA / "Finance" / "FactFinancialTransactions.csv", parse_dates=["TransactionDate"])
df_accounts  = pd.read_csv(DATA / "Finance" / "DimAccounts.csv")
df_costctr   = pd.read_csv(DATA / "Finance" / "DimCostCenters.csv")
df_payroll   = pd.read_csv(DATA / "HR" / "FactPayroll.csv", parse_dates=["PayDate"])
df_employees = pd.read_csv(DATA / "HR" / "DimEmployees.csv")
df_departments = pd.read_csv(DATA / "HR" / "DimDepartments.csv")

# Enrich
df_orders = df_orders.merge(df_books[["BookID", "Genre"]], on="BookID", how="left")
df_finance = df_finance.merge(df_accounts[["AccountID", "AccountType"]], on="AccountID", how="left")
df_budget = df_budget.merge(df_accounts[["AccountID", "AccountType", "AccountName"]], on="AccountID", how="left")
df_budget = df_budget.merge(df_costctr[["CostCenterID", "CostCenterName", "Department"]], on="CostCenterID", how="left")

print(f"Orders:     {len(df_orders):>6,} rows  ({df_orders['OrderDate'].min().date()} – {df_orders['OrderDate'].max().date()})")
print(f"Budget:     {len(df_budget):>6,} rows")
print(f"Finance:    {len(df_finance):>6,} rows")
print(f"Payroll:    {len(df_payroll):>6,} rows")
print(f"Employees:  {len(df_employees):>6,} rows")
print(f"Accounts:   {len(df_accounts):>6,} rows")

In [ ]:
# Cell 3: Helper functions

def generate_plan_months(start_date, horizon):
    """Generate a list of monthly plan dates starting from start_date."""
    return [start_date + relativedelta(months=i) for i in range(horizon)]


def apply_growth_scenario(base_values, growth_rate, months):
    """Apply compounding monthly growth to base values."""
    monthly_rate = (1 + growth_rate) ** (1/12) - 1
    return [v * (1 + monthly_rate) ** (i + 1) for i, v in enumerate(base_values)]


def classify_variance(variance_pct, thresholds):
    """Classify variance as Favorable, On Track, Unfavorable, or Critical."""
    if variance_pct >= thresholds["favorable"]:
        return "Favorable"
    elif variance_pct >= 0:
        return "On Track"
    elif variance_pct >= thresholds["critical"]:
        return "Unfavorable"
    else:
        return "Critical"


def fiscal_quarter(month_num):
    """Convert month number (1-12) to fiscal quarter string."""
    return f"Q{(month_num - 1) // 3 + 1}"


print("✓ Planning helpers loaded")

---
## 1. Revenue Targets by Channel (Scenario Modeling)

Builds forward-looking revenue targets per channel using historical actuals as the base,
then applies growth assumptions for Base (+8%), Optimistic (+15%), and Conservative (+3%)
scenarios over a 12-month planning horizon.

In [ ]:
# Cell 4: Revenue Targets by Channel

df_orders["OrderMonth"] = df_orders["OrderDate"].dt.to_period("M").dt.to_timestamp()

# Aggregate monthly revenue by channel
monthly_revenue = (
    df_orders.groupby(["OrderMonth", "Channel"])
    .agg(Revenue=("TotalAmount", "sum"), Orders=("OrderID", "count"))
    .reset_index()
)

# Use last 6 months as baseline for planning
last_date = monthly_revenue["OrderMonth"].max()
baseline_start = last_date - pd.DateOffset(months=5)
baseline = monthly_revenue[monthly_revenue["OrderMonth"] >= baseline_start]

channels = monthly_revenue["Channel"].unique()
plan_start = last_date + pd.DateOffset(months=1)
plan_months = generate_plan_months(plan_start, PLANNING_HORIZON)

# Build targets per channel × scenario
revenue_targets = []
for channel in channels:
    ch_baseline = baseline[baseline["Channel"] == channel]["Revenue"].values
    if len(ch_baseline) == 0:
        continue
    # Tile baseline to fill planning horizon
    base_pattern = np.tile(ch_baseline, (PLANNING_HORIZON // len(ch_baseline)) + 1)[:PLANNING_HORIZON]

    for scenario in SCENARIOS:
        targets = apply_growth_scenario(base_pattern, GROWTH[scenario], PLANNING_HORIZON)
        for i, (pm, target) in enumerate(zip(plan_months, targets)):
            revenue_targets.append({
                "PlanMonth": pm,
                "Channel": channel,
                "Scenario": scenario,
                "TargetRevenue": round(target, 2),
                "FiscalYear": f"FY{pm.year}" if pm.month >= 7 else f"FY{pm.year}",
                "FiscalQuarter": fiscal_quarter(pm.month),
                "PlanHorizon": i + 1,
                "RecordType": "Plan",
            })

df_rev_targets = pd.DataFrame(revenue_targets)
print(f"✓ Revenue Targets: {len(df_rev_targets)} rows across {len(channels)} channels × {len(SCENARIOS)} scenarios")
print(f"  Plan range: {plan_months[0].strftime('%b %Y')} – {plan_months[-1].strftime('%b %Y')}")
df_rev_targets.head(10)

In [ ]:
# Cell 5: Visualize Revenue Targets by Channel

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Revenue Targets by Channel — Scenario Comparison", fontsize=14, fontweight="bold")

for idx, channel in enumerate(sorted(channels)[:4]):
    ax = axes[idx // 2, idx % 2]

    # Plot actuals
    ch_actual = monthly_revenue[monthly_revenue["Channel"] == channel]
    ax.plot(ch_actual["OrderMonth"], ch_actual["Revenue"],
            color="#393E41", linewidth=2, label="Actual")

    # Plot each scenario
    for scenario in SCENARIOS:
        ch_plan = df_rev_targets[
            (df_rev_targets["Channel"] == channel) &
            (df_rev_targets["Scenario"] == scenario)
        ]
        ax.plot(ch_plan["PlanMonth"], ch_plan["TargetRevenue"],
                color=SCENARIO_COLORS[scenario], linewidth=2,
                linestyle="--", marker="o", markersize=3, label=scenario)

    ax.set_title(f"{channel}", fontsize=11, fontweight="bold")
    ax.set_ylabel("Revenue ($)")
    ax.legend(loc="upper left", fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

---
## 2. Financial Plan — P&L Targets with Scenario Modeling

Creates a planning model that bridges the existing FactBudget data with forward-looking
financial targets. Each account type gets Base/Optimistic/Conservative scenarios derived
from historical budget patterns.

In [ ]:
# Cell 6: Financial Plan by Account Type

# Aggregate budget by month + account type
budget_by_type = (
    df_budget.groupby(["FiscalYear", "FiscalMonth", "AccountType"])
    .agg(
        BudgetAmount=("BudgetAmount", "sum"),
        ActualAmount=("ActualAmount", "sum"),
    )
    .reset_index()
)

# Derive planning baseline from last fiscal year of data
last_fy = budget_by_type["FiscalYear"].max()
baseline_budget = budget_by_type[budget_by_type["FiscalYear"] == last_fy]

account_types = baseline_budget["AccountType"].unique()

financial_targets = []
for acct_type in account_types:
    acct_baseline = baseline_budget[baseline_budget["AccountType"] == acct_type]
    monthly_budget = acct_baseline.sort_values("FiscalMonth")["BudgetAmount"].values
    if len(monthly_budget) == 0:
        continue
    base_pattern = np.tile(monthly_budget, (PLANNING_HORIZON // len(monthly_budget)) + 1)[:PLANNING_HORIZON]

    for scenario in SCENARIOS:
        targets = apply_growth_scenario(base_pattern, GROWTH[scenario], PLANNING_HORIZON)
        for i, (pm, target) in enumerate(zip(plan_months, targets)):
            financial_targets.append({
                "PlanMonth": pm,
                "AccountType": acct_type,
                "Scenario": scenario,
                "PlannedAmount": round(target, 2),
                "FiscalYear": f"FY{pm.year}",
                "FiscalQuarter": fiscal_quarter(pm.month),
                "PlanHorizon": i + 1,
                "RecordType": "Plan",
            })

df_fin_targets = pd.DataFrame(financial_targets)
print(f"✓ Financial Plan: {len(df_fin_targets)} rows across {len(account_types)} account types × {len(SCENARIOS)} scenarios")
print(f"  Account types: {', '.join(account_types)}")
df_fin_targets.head(10)

In [ ]:
# Cell 7: Visualize Financial Plan by Account Type

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Financial Plan by Account Type — Scenario Comparison", fontsize=14, fontweight="bold")

for idx, acct_type in enumerate(sorted(account_types)[:4]):
    ax = axes[idx // 2, idx % 2]
    for scenario in SCENARIOS:
        plan_data = df_fin_targets[
            (df_fin_targets["AccountType"] == acct_type) &
            (df_fin_targets["Scenario"] == scenario)
        ]
        ax.plot(plan_data["PlanMonth"], plan_data["PlannedAmount"],
                color=SCENARIO_COLORS[scenario], linewidth=2,
                linestyle="--", marker="o", markersize=3, label=scenario)

    # Overlay actuals from budget
    acct_actuals = budget_by_type[budget_by_type["AccountType"] == acct_type]
    if not acct_actuals.empty:
        ax.axhline(y=acct_actuals["ActualAmount"].mean(), color="#393E41",
                   linestyle=":", alpha=0.5, label="Avg Actual")

    ax.set_title(f"{acct_type}", fontsize=11, fontweight="bold")
    ax.set_ylabel("Amount ($)")
    ax.legend(loc="best", fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

---
## 3. Workforce Plan — Headcount & Payroll Targets

Generates workforce planning targets per department using payroll history as the baseline.
Each scenario models different headcount growth trajectories.

In [ ]:
# Cell 8: Workforce Plan by Department

df_payroll = df_payroll.merge(df_employees[["EmployeeID", "DepartmentID"]], on="EmployeeID", how="left")
df_payroll = df_payroll.merge(df_departments[["DepartmentID", "DepartmentName"]], on="DepartmentID", how="left")
df_payroll["PayMonth"] = df_payroll["PayDate"].dt.to_period("M").dt.to_timestamp()

monthly_workforce = (
    df_payroll.groupby(["PayMonth", "DepartmentName"])
    .agg(
        Headcount=("EmployeeID", "nunique"),
        TotalPayroll=("NetPay", "sum"),
    )
    .reset_index()
)

departments = monthly_workforce["DepartmentName"].dropna().unique()

# Workforce growth multipliers (headcount grows slower than payroll)
HC_GROWTH = {"Base": 0.05, "Optimistic": 0.10, "Conservative": 0.02}

workforce_targets = []
for dept in departments:
    dept_data = monthly_workforce[monthly_workforce["DepartmentName"] == dept].sort_values("PayMonth")
    if dept_data.empty:
        continue

    # Use last available values as baseline
    last_hc = dept_data["Headcount"].iloc[-1]
    last_payroll = dept_data["TotalPayroll"].iloc[-1]

    for scenario in SCENARIOS:
        for i, pm in enumerate(plan_months):
            monthly_hc_rate = (1 + HC_GROWTH[scenario]) ** (1/12) - 1
            monthly_pay_rate = (1 + GROWTH[scenario]) ** (1/12) - 1

            planned_hc = max(1, round(last_hc * (1 + monthly_hc_rate) ** (i + 1)))
            planned_payroll = round(last_payroll * (1 + monthly_pay_rate) ** (i + 1), 2)

            workforce_targets.append({
                "PlanMonth": pm,
                "Department": dept,
                "Scenario": scenario,
                "PlannedHeadcount": planned_hc,
                "PlannedPayroll": planned_payroll,
                "FiscalYear": f"FY{pm.year}",
                "FiscalQuarter": fiscal_quarter(pm.month),
                "PlanHorizon": i + 1,
                "RecordType": "Plan",
            })

df_wf_targets = pd.DataFrame(workforce_targets)
print(f"✓ Workforce Plan: {len(df_wf_targets)} rows across {len(departments)} departments × {len(SCENARIOS)} scenarios")
print(f"  Departments: {', '.join(sorted(departments))}")
df_wf_targets.head(10)

In [ ]:
# Cell 9: Visualize Workforce Plan

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Workforce Plan — Headcount & Payroll Targets", fontsize=14, fontweight="bold")

# Total headcount by scenario
ax = axes[0]
for scenario in SCENARIOS:
    plan = df_wf_targets[df_wf_targets["Scenario"] == scenario]
    monthly_hc = plan.groupby("PlanMonth")["PlannedHeadcount"].sum().reset_index()
    ax.plot(monthly_hc["PlanMonth"], monthly_hc["PlannedHeadcount"],
            color=SCENARIO_COLORS[scenario], linewidth=2, label=scenario)
# Overlay actual total headcount
actual_hc = monthly_workforce.groupby("PayMonth")["Headcount"].sum().reset_index()
ax.plot(actual_hc["PayMonth"], actual_hc["Headcount"],
        color="#393E41", linewidth=2, label="Actual")
ax.set_title("Total Headcount", fontweight="bold")
ax.set_ylabel("Headcount")
ax.legend(fontsize=8)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.tick_params(axis="x", rotation=45)

# Total payroll by scenario
ax = axes[1]
for scenario in SCENARIOS:
    plan = df_wf_targets[df_wf_targets["Scenario"] == scenario]
    monthly_pay = plan.groupby("PlanMonth")["PlannedPayroll"].sum().reset_index()
    ax.plot(monthly_pay["PlanMonth"], monthly_pay["PlannedPayroll"],
            color=SCENARIO_COLORS[scenario], linewidth=2, label=scenario)
actual_pay = monthly_workforce.groupby("PayMonth")["TotalPayroll"].sum().reset_index()
ax.plot(actual_pay["PayMonth"], actual_pay["TotalPayroll"],
        color="#393E41", linewidth=2, label="Actual")
ax.set_title("Total Payroll", fontweight="bold")
ax.set_ylabel("Payroll ($)")
ax.legend(fontsize=8)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

---
## 4. Plan vs Actual — Variance Analysis

Consolidates budget vs actual data from FactBudget to produce a variance analysis table
with status flags (Favorable, On Track, Unfavorable, Critical) — the core capability
that Planning in Fabric IQ enables natively.

In [ ]:
# Cell 10: Plan vs Actual Variance Analysis

# Build from existing FactBudget
variance_records = []

# --- Revenue domain: Budget vs Actual by month ---
revenue_budget = df_budget[df_budget["AccountType"] == "Revenue"].copy()
for _, row in revenue_budget.iterrows():
    var_pct = row["VariancePct"] / 100 if pd.notna(row["VariancePct"]) else 0
    variance_records.append({
        "Domain": "Revenue",
        "Category": row.get("AccountName", "Revenue"),
        "FiscalYear": row["FiscalYear"],
        "FiscalQuarter": row["FiscalQuarter"],
        "FiscalMonth": row["FiscalMonth"],
        "PlannedAmount": row["BudgetAmount"],
        "ActualAmount": row["ActualAmount"],
        "Variance": row["Variance"],
        "VariancePct": var_pct,
        "Status": classify_variance(var_pct, VARIANCE_THRESHOLDS),
    })

# --- Finance domain: Expense accounts ---
expense_budget = df_budget[df_budget["AccountType"] != "Revenue"].copy()
for _, row in expense_budget.iterrows():
    var_pct = row["VariancePct"] / 100 if pd.notna(row["VariancePct"]) else 0
    # For expenses, negative variance is favorable (underspent)
    adjusted_var = -var_pct  # flip for expense items
    variance_records.append({
        "Domain": "Finance",
        "Category": row.get("AccountName", row["AccountType"]),
        "FiscalYear": row["FiscalYear"],
        "FiscalQuarter": row["FiscalQuarter"],
        "FiscalMonth": row["FiscalMonth"],
        "PlannedAmount": row["BudgetAmount"],
        "ActualAmount": row["ActualAmount"],
        "Variance": row["Variance"],
        "VariancePct": var_pct,
        "Status": classify_variance(adjusted_var, VARIANCE_THRESHOLDS),
    })

df_variance = pd.DataFrame(variance_records)

print(f"✓ Variance Analysis: {len(df_variance)} rows")
print(f"\nStatus distribution:")
print(df_variance["Status"].value_counts().to_string())
print(f"\nDomain breakdown:")
print(df_variance.groupby("Domain")["Variance"].agg(["count", "sum", "mean"]).to_string())

In [ ]:
# Cell 11: Visualize Variance Analysis

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Plan vs Actual — Variance Analysis", fontsize=14, fontweight="bold")

# 1. Status distribution
ax = axes[0]
status_counts = df_variance["Status"].value_counts()
status_colors = {"Favorable": "#44BBA4", "On Track": "#2E86AB",
                 "Unfavorable": "#F18F01", "Critical": "#E94F37"}
bars = ax.bar(status_counts.index, status_counts.values,
              color=[status_colors.get(s, "#999") for s in status_counts.index])
ax.set_title("Variance Status Distribution", fontweight="bold")
ax.set_ylabel("Count")
for bar, val in zip(bars, status_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(val), ha="center", fontweight="bold")

# 2. Variance by domain
ax = axes[1]
domain_var = df_variance.groupby("Domain")["Variance"].sum().reset_index()
bars = ax.bar(domain_var["Domain"], domain_var["Variance"],
              color=[DOMAIN_COLORS.get(d, "#999") for d in domain_var["Domain"]])
ax.set_title("Total Variance by Domain", fontweight="bold")
ax.set_ylabel("Variance ($)")
ax.axhline(y=0, color="#393E41", linewidth=0.8)

# 3. Quarterly variance trend
ax = axes[2]
quarterly = df_variance.groupby(["FiscalQuarter", "Domain"])["Variance"].sum().unstack(fill_value=0)
quarterly.plot(kind="bar", ax=ax, color=[DOMAIN_COLORS.get(d, "#999") for d in quarterly.columns])
ax.set_title("Quarterly Variance by Domain", fontweight="bold")
ax.set_ylabel("Variance ($)")
ax.legend(fontsize=8)
ax.axhline(y=0, color="#393E41", linewidth=0.8)
ax.tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

---
## 5. Scenario Comparison Summary

Aggregates all planning models into a single executive summary table for side-by-side
scenario comparison — designed for Planning in Fabric IQ dashboards and collaborative
review workflows.

In [ ]:
# Cell 12: Scenario Comparison Summary

# Revenue scenario summary
rev_summary = (
    df_rev_targets.groupby(["Scenario", "FiscalQuarter"])
    .agg(PlannedTotal=("TargetRevenue", "sum"))
    .reset_index()
)
rev_summary["Domain"] = "Revenue"

# Financial scenario summary
fin_summary = (
    df_fin_targets.groupby(["Scenario", "FiscalQuarter"])
    .agg(PlannedTotal=("PlannedAmount", "sum"))
    .reset_index()
)
fin_summary["Domain"] = "Finance"

# Workforce scenario summary (payroll costs)
wf_summary = (
    df_wf_targets.groupby(["Scenario", "FiscalQuarter"])
    .agg(PlannedTotal=("PlannedPayroll", "sum"))
    .reset_index()
)
wf_summary["Domain"] = "Workforce"

df_scenario_summary = pd.concat([rev_summary, fin_summary, wf_summary], ignore_index=True)
df_scenario_summary["RecordType"] = "ScenarioSummary"

print(f"✓ Scenario Summary: {len(df_scenario_summary)} rows")
print()

# Pivot for display
pivot = df_scenario_summary.pivot_table(
    index=["Domain", "FiscalQuarter"], columns="Scenario",
    values="PlannedTotal", aggfunc="sum"
).round(0)
print(pivot.to_string())

In [ ]:
# Cell 13: Visualize Scenario Comparison

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Executive Scenario Comparison — All Domains", fontsize=14, fontweight="bold")

for idx, domain in enumerate(["Revenue", "Finance", "Workforce"]):
    ax = axes[idx]
    domain_data = df_scenario_summary[df_scenario_summary["Domain"] == domain]

    x_labels = sorted(domain_data["FiscalQuarter"].unique())
    x = np.arange(len(x_labels))
    width = 0.25

    for i, scenario in enumerate(SCENARIOS):
        vals = [domain_data[
            (domain_data["Scenario"] == scenario) &
            (domain_data["FiscalQuarter"] == q)
        ]["PlannedTotal"].sum() for q in x_labels]
        ax.bar(x + i * width, vals, width,
               label=scenario, color=SCENARIO_COLORS[scenario])

    ax.set_title(f"{domain}", fontsize=11, fontweight="bold")
    ax.set_ylabel("Amount ($)")
    ax.set_xticks(x + width)
    ax.set_xticklabels(x_labels)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## Summary

This notebook demonstrates the planning concepts enabled by **Planning in Fabric IQ**:

| Capability | Horizon Books Implementation |
|-----------|-----------------------------|
| **Integrated Budgeting** | Revenue, Financial, and Workforce targets built on governed Fabric data |
| **Scenario Modeling** | Base (+8%), Optimistic (+15%), Conservative (+3%) across all domains |
| **Plan vs Actual** | Variance analysis with status flags (Favorable/On Track/Unfavorable/Critical) |
| **Writeback-Ready** | All planning tables designed for Fabric SQL writeback operations |
| **Collaborative Planning** | Scenario comparison summaries for executive review and approval workflows |
| **Semantic Integration** | Planning tables connect to Power BI semantic models for real-time dashboards |

In production, these tables are created by the Fabric notebook (`notebooks/04_Forecasting.py`
planning extension) and consumed by Planning in Fabric IQ for interactive budget editing,
scenario comparison, and approval workflows — all within the governed Fabric environment.

Reference: [Planning in Fabric IQ announcement](https://blog.fabric.microsoft.com/en-us/blog/introducing-planning-in-microsoft-fabric-iq-from-historical-data-to-forecasting-the-future)